# Gold: Star Schema

### 1.0: Configuration

In [0]:
%run ./00_utils

### 1.1: Upsert Helper

All five gold tables follow the same load pattern: create on the first run, MERGE on every run after. Defining it once keeps each table cell down to "build the SELECT, call `upsert`".

In [ ]:
def upsert(source_df, table_name, keys, update_matched=True):
    """
    Create the gold table on the first run, MERGE into it on every run after.

    Args:
        source_df: DataFrame holding the current version of the data
        table_name: target table name in the gold schema
        keys: business key columns that uniquely identify a row
        update_matched: True for dimensions, whose attributes may change;
                        False for immutable rows (facts, dates): insert-only
    """
    table = f"{CATALOG}.gold.{table_name}"

    if not spark.catalog.tableExists(table):
        source_df.write.saveAsTable(table)
    else:
        source_df.createOrReplaceTempView("source")
        on_clause = " AND ".join(f"target.{k} = source.{k}" for k in keys)
        update_clause = "WHEN MATCHED THEN UPDATE SET *" if update_matched else ""
        spark.sql(f"""
            MERGE INTO {table} AS target
            USING source AS source
            ON {on_clause}
            {update_clause}
            WHEN NOT MATCHED THEN INSERT *
        """)

    print(f"{table_name}: {spark.table(table).count()} rows")

### 2.0: Dimension: Locations

In [ ]:
# One row per monitoring station. Attributes (name, owner, coordinates) can
# change over time, so matched rows are updated in place (SCD Type 1).
dim_locations = spark.sql(f"""
    SELECT 
        id as location_id,
        name as location_name,
        locality,
        latitude,
        longitude,
        country_code,
        country_name,
        owner_name,
        provider_name,
        timezone
    FROM {CATALOG}.silver.locations
""")

upsert(dim_locations, "dim_locations", keys=["location_id"])

### 3.0: Dimension: Parameters

In [ ]:
# One row per pollutant (PM2.5, O3, ...), used to label and filter measurements.
dim_parameters = spark.sql(f"""
    SELECT 
        parameter_id,
        parameter_name,
        parameter_units,
        parameter_display_name
    FROM {CATALOG}.silver.parameters
""")

upsert(dim_parameters, "dim_parameters", keys=["parameter_id"])

### 4.0: Dimension: Sensors

In [ ]:
# One row per physical sensor. DISTINCT because silver.sensors also carries
# the sensor-to-location mapping, which is not needed at this grain.
dim_sensors = spark.sql(f"""
    SELECT DISTINCT
        sensor_id,
        sensor_name
    FROM {CATALOG}.silver.sensors
""")

upsert(dim_sensors, "dim_sensors", keys=["sensor_id"])

### 5.0: Dimension: Date

In [ ]:
# Calendar dimension: a complete, gap-free date range, so days with zero
# readings still exist in the model (key signal for sensor health).
# Range kept small on purpose: this is a demo project.
# A date row never changes once written, so the load is insert-only.
START_DATE = "2025-01-01"
END_DATE = "2027-12-31"

dim_date = spark.sql(f"""
    SELECT
        date_id,
        YEAR(date_id) as year,
        MONTH(date_id) as month,
        DAY(date_id) as day,
        DAYOFWEEK(date_id) as day_of_week,
        DAYNAME(date_id) as day_name,
        WEEKOFYEAR(date_id) as week_of_year,
        QUARTER(date_id) as quarter
    FROM (
        SELECT explode(sequence(DATE'{START_DATE}', DATE'{END_DATE}', INTERVAL 1 DAY)) as date_id
    )
""")

upsert(dim_date, "dim_date", keys=["date_id"], update_matched=False)

### 6.0: Fact: Measurements

In [ ]:
# One row per sensor reading: the grain of the star schema.
# The join to silver.sensors adds parameter_id. INNER on purpose: a reading
# whose sensor has no metadata cannot be attributed to a pollutant.
# Facts are immutable events, so the load is insert-only: the business key
# (sensor_id, datetime_utc) makes re-runs idempotent.
fact_measurements = spark.sql(f"""
    SELECT 
        m.sensors_id as sensor_id,
        m.locations_id as location_id,
        s.parameter_id,
        CAST(DATE(m.datetime_utc) AS DATE) as date_id,
        m.datetime_utc,
        m.value,
        m.ingested_at
    FROM {CATALOG}.silver.measurements m
    JOIN {CATALOG}.silver.sensors s ON m.sensors_id = s.sensor_id
""")

upsert(fact_measurements, "fact_measurements", keys=["sensor_id", "datetime_utc"], update_matched=False)

### 7.0: Verify Star Schema

In [ ]:
# Sanity check: row counts per gold table (visible in the job run output)
spark.sql(f"""
    SELECT 'dim_locations' as table_name, COUNT(*) as rows FROM {CATALOG}.gold.dim_locations
    UNION ALL
    SELECT 'dim_parameters', COUNT(*) FROM {CATALOG}.gold.dim_parameters
    UNION ALL
    SELECT 'dim_sensors', COUNT(*) FROM {CATALOG}.gold.dim_sensors
    UNION ALL
    SELECT 'dim_date', COUNT(*) FROM {CATALOG}.gold.dim_date
    UNION ALL
    SELECT 'fact_measurements', COUNT(*) FROM {CATALOG}.gold.fact_measurements
""").display()